# Pipeline Quickstart: Bronze → Silver → Gold with DAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakelogic/LakeLogic/blob/main/examples/02_pipeline_quickstart/02_pipeline_quickstart.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/02_pipeline_quickstart/02_pipeline_quickstart.ipynb)

## Business Scenario

You need to build a **medallion lakehouse pipeline** that:

1. **Bronze** — Ingest raw employee CSV with minimal validation
2. **Silver** — Cleanse, enrich with tenure categories, enforce business rules
3. **Gold** — Aggregate into a department headcount summary for dashboards

LakeLogic does this with **YAML contracts** and a **single `pipeline.run()` call**.

---

## What You'll Learn

| Section | Feature |
|---|---|
| 1 | `_system.yaml` — the pipeline registry |
| 2 | DAG visualization — see contract dependencies |
| 3 | Pipeline execution — bronze → silver → gold in one call |
| 4 | Per-layer controls — schema evolution, materialization defaults |
| 5 | Run summary — what happened across the full pipeline |


## Setup

In [ ]:
import importlib.util
import sys
import os
from pathlib import Path

if importlib.util.find_spec("lakelogic") is None:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "lakelogic", "-q"], check=True)
    print("lakelogic installed.")
else:
    print("lakelogic ready.")

# Ensure we're in the right directory
if "google.colab" in sys.modules:
    repo = Path("/content/LakeLogic")
    if not repo.exists():
        import subprocess
        subprocess.run(["git", "clone", "--quiet", "https://github.com/lakelogic/LakeLogic.git", str(repo)], check=True)
    os.chdir(repo / "examples" / "02_pipeline_quickstart")
    print(f"Working directory: {Path.cwd()}")

print("Setup complete.")

---

## 1. The Pipeline Registry (`_system.yaml`)

A `_system.yaml` file defines your entire pipeline:

- **Contracts** — which entities to process at each layer
- **Dependencies** — `depends_on` determines execution order
- **Per-layer defaults** — schema evolution, materialization strategy

```yaml
contracts:
  - layer: bronze
    entity: raw_employees
    path: contracts/bronze/bronze_employees_v1.0.yaml

  - layer: silver
    entity: clean_employees
    path: contracts/silver/silver_employees_v1.0.yaml
    depends_on: [raw_employees]    # ← waits for bronze

  - layer: gold
    entity: headcount
    path: contracts/gold/gold_headcount_v1.0.yaml
    depends_on: [clean_employees]  # ← waits for silver

server_defaults:
  bronze:
    schema_evolution: append      # source schemas may change
  silver:
    schema_evolution: strict      # curated — no surprises
  gold:
    schema_evolution: strict      # business layer — locked down
```

> **Key rule:** Each entity name must be unique across the pipeline.
> `depends_on` references entity names, not layer names.

Let's load it:


In [ ]:
from lakelogic.core.registry import DomainRegistry
from lakelogic.pipeline import LakehousePipeline

registry = DomainRegistry.from_yaml("_system.yaml")

print(f"Domain  : {registry.domain}")
print(f"System  : {registry.system}")
print(f"Contracts: {len(registry.contracts)}")
print()
for c in registry.contracts:
    deps = c.depends_on or []
    print(f"  [{c.layer}] {c.entity} {'→ depends_on: ' + str(deps) if deps else '(root)'}")

---

## 2. DAG Visualization

LakeLogic resolves contract dependencies into an execution DAG.
Visualize it to see the processing order:


In [6]:
pipeline = LakehousePipeline(registry, engine="polars")

# Generate the DAG as interactive HTML
dag_html = pipeline.visualize_dag()

if dag_html:
    from IPython.display import HTML
    display(HTML(dag_html))
    print("\n↑ Interactive DAG — hover over nodes for details.")
else:
    # Fallback: text-based DAG
    print("DAG Execution Order:")
    print("  bronze.raw_employees  →  silver.clean_employees  →  gold.headcount")
    print("  (ingest raw)             (cleanse+enrich)           (aggregate)")


↑ Interactive DAG — hover over nodes for details.


---

## 3. Pipeline Execution

Run the entire medallion pipeline with a single call.
LakeLogic processes contracts in DAG order: bronze → silver → gold.


In [ ]:
# ── Run all layers ────────────────────────────────────────────────────
summary = pipeline.run(
    target_layers="bronze,silver,gold",
    environment="dev",
)

# ── Display summary ───────────────────────────────────────────────────
print("\n" + "=" * 65)
print("PIPELINE RUN SUMMARY")
print("=" * 65)
print(f"  Pipeline run : {summary.run_id}")
print(f"  Environment  : {summary.environment}")
print()
print(f"  {'Contract':<30} {'Layer':<8} {'Status':<15} Rows")
print(f"  {'-'*30} {'-'*8} {'-'*15} ----")

for r in summary.results:
    contract = r.get("contract", "?")
    layer = r.get("layer", "?")
    status = r.get("status", "?")
    rows = r.get("rows", "-")
    error = r.get("error", "")
    print(f"  {contract:<30} {layer:<8} {status:<15} {rows}")
    if error:
        print(f"    ↳ {error}")

print("=" * 65)

---

## 4. Per-Layer Controls

LakeLogic applies different behaviors per layer, all configured in `_system.yaml`:

| Layer | Schema Evolution | Materialization | Purpose |
|---|---|---|---|
| **Bronze** | `append` — auto-add new columns | `append` — never lose raw data | Fidelity |
| **Silver** | `strict` — fail on schema change | `overwrite` — always fresh | Quality |
| **Gold** | `strict` — locked down | `overwrite` — rebuild aggregates | Trust |

### Schema Evolution Options

```yaml
server_defaults:
  bronze:
    schema_evolution: append      # auto-add new columns (Delta mergeSchema)
    allow_schema_drift: true      # log drift but don't fail
    cast_to_string: false         # set true for 'all strings' bronze pattern
  silver:
    schema_evolution: strict      # fail on schema mismatch (production safety)
  gold:
    schema_evolution: strict
```

Individual contracts can **override** any server default:

```yaml
# In a specific contract:
server:
  type: table
  path: "catalog.schema.table"
  schema_evolution: merge    # ← overrides the layer default
```


---

## 5. Inspect the Output

Let's verify what was materialized at each layer:


In [ ]:
output_dir = Path("output")

if output_dir.exists():
    for layer_dir in sorted(output_dir.iterdir()):
        if layer_dir.is_dir():
            files = list(layer_dir.rglob("*"))
            data_files = [f for f in files if f.is_file() and not f.name.startswith(".")]
            total_size = sum(f.stat().st_size for f in data_files)
            print(f"📁 {layer_dir.name}")
            print(f"   Files: {len(data_files)} | Size: {total_size:,} bytes")
            for f in data_files[:3]:
                print(f"   └── {f.name}")
            print()
else:
    print("No output directory — run the pipeline first.")

---

## Pipeline Features Reference

### Selective Layer Runs

```python
# Run only bronze
pipeline.run(target_layers="bronze")

# Run silver + gold (skip bronze)
pipeline.run(target_layers="silver,gold")

# Filter to specific entities
pipeline.run(target_layers="bronze", entity_filter="raw_employees")
```

### Reprocessing

Fix historical data without rebuilding the full pipeline:

```python
pipeline.run(
    target_layers="silver",
    reprocess_from="2024-01-01",
    reprocess_to="2024-03-31",
    reprocess_column="hire_date",
)
```

### Reset & Reload

```python
# DROP and recreate tables (dev only)
pipeline.run(reset_layers="silver,gold")

# TRUNCATE tables but keep structure
pipeline.run(reload_layers="silver")
```

### Dry Run

```python
# Validate contracts without writing anything
pipeline.run(dry_run=True)
```

### GDPR / Right to Delete

```python
pipeline.run(
    forget_column="email",
    forget_values="frank@company.com",
    forget_strategy="soft_delete",  # or 'hard_delete' or 'hash'
)
```

---

## Summary

| Concept | What You Saw |
|---|---|
| **`_system.yaml`** | Single file defines contracts, dependencies, per-layer defaults |
| **DAG** | Visual dependency graph — LakeLogic resolves execution order |
| **`pipeline.run()`** | One call processes all layers in order |
| **Server defaults** | Schema evolution, drift controls per layer |
| **Reprocessing** | Fix historical data surgically |

### Next Steps

- **Scale to Databricks** — Deploy with `databricks.yml` bundles
- **Add SCD2** — Track slowly changing dimensions
- **External logic** — Custom Python processors for gold aggregations
- **Parallel execution** — `pipeline.run(parallel=True, max_workers=4)`
